# 05 — Chuẩn hóa và mã hóa biến phân loại

**Mục tiêu:** so sánh các cách chuẩn hóa thuộc tính số và mã hóa thuộc tính phân loại.

**Rubric:** Bảng — phát hiện ngoại lai và mã hóa (10%, cùng notebook 04).

**Đề yêu cầu:**
- [ ] Chuẩn hóa: Min-Max, Z-score, Robust Scaling, Quantile Transform (uniform hoặc normal).
- [ ] Kiểm định Levene đánh giá đồng nhất phương sai; violin plot theo thuộc tính.
- [ ] Mã hóa cơ bản: One-Hot, Ordinal.
- [ ] Mã hóa nâng cao: Target Encoding có cross-validation chống leakage, Binary Encoding (biến > 20 giá trị), Frequency Encoding.
- [ ] Đo tỷ lệ phương sai giải thích và VIF để phát hiện đa cộng tuyến mới.

**Đầu vào:** development set đã điền thiếu (03) và xử lý ngoại lai (04).

**Đầu ra:** `results/tabular/tables/scaling_levene.csv`, `encoding_comparison.csv`, violin plot; lựa chọn scaler/encoder.
---
*Quy ước khung:* ô **[Code]** sẽ được thay bằng ô code; ô **[Nhận xét]** chỉ viết sau khi có kết quả thật. Mọi score/p-value phải được **in ra và gán vào biến Python** (yêu cầu của đề).

## 0. Thiết lập

> **[Code]** Import thư viện; tìm gốc repo (`find_project_root`, nên chuyển vào `src/data_utils.py`); đọc `data/raw/home_credit/application_train.csv`; nạp `data/splits/home_credit_split_ids.csv` và **chỉ lấy phần development**. Không dùng phần test trong notebook này. `SEED = 42`. Áp dụng các lựa chọn của notebook 03–04 bên trong pipeline.

# Phần A — Chuẩn hóa thuộc tính số

## A1. Min-Max, Z-score, Robust, Quantile

**Lý thuyết — cần trình bày:**
- Min-Max: $x' = \frac{x - x_{min}}{x_{max} - x_{min}}$ — nhạy với ngoại lai.
- Z-score: $x' = \frac{x - \mu}{\sigma}$ — đổi vị trí/thang đo, **không** làm phân phối lệch trở thành chuẩn.
- Robust: $x' = \frac{x - \text{median}}{IQR}$.
- Quantile: ánh xạ qua hàm phân phối thực nghiệm sang uniform hoặc normal — biến đổi phi tuyến.

> **[Code]** Fit từng scaler trên phần train; lưu bản biến đổi của vài cột tiêu biểu cho mỗi phương pháp.

> **[Nhận xét]** Phương pháp nào bị ảnh hưởng bởi ngoại lai? Thứ hạng giữa các điểm có thay đổi không?

## A2. Violin plot và kiểm định Levene

**Lý thuyết — cần trình bày:**
- Levene: $H_0$ — phương sai các nhóm bằng nhau; dựa trên độ lệch tuyệt đối so với trung vị/trung bình.
- **Cần nêu rõ thiết kế:** so sánh phương sai giữa các nhóm nào (ví dụ giữa các thuộc tính sau cùng một scaler, hoặc giữa hai lớp TARGET).

> **[Code]** Violin plot từng thuộc tính sau mỗi scaler; chạy Levene theo thiết kế đã nêu, lưu vào biến `levene_results`.

> **[Nhận xét]** Scaler nào đưa các thuộc tính về thang đo đồng nhất hơn? Hệ quả cho KNN/LOF/mô hình tuyến tính.

# Phần B — Mã hóa thuộc tính phân loại

## B1. Khảo sát thuộc tính phân loại

**Lý thuyết — cần trình bày:**
- Biến danh nghĩa và biến thứ bậc; số nhóm (cardinality) quyết định cách mã hóa.

> **[Code]** Bảng các cột phân loại: số nhóm, tỷ lệ thiếu, nhóm hiếm; xác định cột > 20 giá trị (ví dụ `ORGANIZATION_TYPE`).

> **[Nhận xét]** Cột nào có thứ tự tự nhiên (ví dụ `NAME_EDUCATION_TYPE`)? Cột nào nhiều nhóm?

## B2. One-Hot và Ordinal

**Lý thuyết — cần trình bày:**
- One-Hot: một cột 0/1 cho mỗi nhóm; `drop="first"` và bẫy biến giả.
- Ordinal: ánh xạ nhóm → số nguyên; chỉ hợp lý khi có thứ tự thật.

> **[Code]** Mã hóa bằng `OneHotEncoder` và `OrdinalEncoder` (thứ tự khai báo tường minh cho cột thứ bậc).

> **[Nhận xét]** Số chiều tăng bao nhiêu với One-Hot?

## B3. Target, Binary, Frequency Encoding

**Lý thuyết — cần trình bày:**
- Target (làm trơn): $\text{enc}(c) = \frac{n_c \bar y_c + m\, \bar y}{n_c + m}$; phải tính **ngoài fold** (cross-fitting) để không dùng nhãn của chính dòng đó — `sklearn.preprocessing.TargetEncoder` làm sẵn việc này.
- Binary: mã thứ tự → các bit nhị phân, cần $\lceil \log_2 K \rceil$ cột — có thể tự cài đặt.
- Frequency: thay nhóm bằng tần suất xuất hiện trong train.

> **[Code]** Cài đặt 3 cách mã hóa cho các cột phù hợp; kiểm tra leakage bằng cách so sánh điểm in-fold và out-of-fold.

> **[Nhận xét]** Vì sao Target Encoding không làm ngoài fold lại cho điểm CV đẹp giả tạo?

## B4. Phương sai giải thích, VIF và điểm mô hình

**Lý thuyết — cần trình bày:**
- VIF: $VIF_j = \frac{1}{1 - R_j^2}$ — đo đa cộng tuyến; VIF > 10 thường là cao.
- Tỷ lệ phương sai giải thích (PCA trên ma trận đã mã hóa) — khác với VIF.

> **[Code]** Với mỗi cách mã hóa: số chiều, số thành phần PCA cho 95% phương sai, VIF của các cột mới, điểm 5-fold CV cùng một mô hình; gom vào biến `encoding_results`.

> **[Nhận xét]** Cách mã hóa nào tạo đa cộng tuyến mới? Chọn cách mã hóa cho từng cột.

## Tổng kết và lựa chọn

> **[Nhận xét]** Scaler và encoder được chọn (có thể khác nhau theo cột) cho pipeline cuối.

## Hạn chế

> **[Nhận xét]** Thiết kế Levene, VIF trên dữ liệu nhiều chiều, phụ thuộc vào mô hình dùng để đánh giá.